# 🐍 Python Mastery for AI Engineering

---

## 1. Overview

Python is the undisputed language of AI. But the Python required for AI engineering in 2026 goes far beyond basic syntax. You need mastery of **type hints**, **async/await**, **decorators**, **generators**, and **data classes** — tools used daily in production AI codebases like LangChain, vLLM, and HuggingFace Transformers.

This notebook focuses on the Python features that separate beginner code from production-quality AI systems.

## 2. Learning Objectives

By the end of this notebook, you will be able to:

- Write fully type-hinted Python functions using the `typing` module
- Use `dataclasses` and `Pydantic` patterns for structured configs
- Write decorators for logging, timing, and retry logic
- Build generators for memory-efficient data pipelines
- Use `asyncio` for concurrent API calls (critical for agentic workflows)
- Apply comprehensions, context managers, and functional patterns

## 3. Imports

In [38]:
from __future__ import annotations

import asyncio
import functools
import time
import json
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Generator, Iterator

print(f"Python version: {sys.version}")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 4. Configuration

In [ ]:
# No special configuration needed for this notebook

## 5. Theory & Implementation

### 5.1 Type Hints — Self-Documenting Code

Modern AI codebases use type hints extensively. They improve readability, enable IDE autocompletion, and catch bugs via `mypy`.

In [ ]:
# without type hints — ambiguous
def process(data, threshold, labels):
    results = []
    for item, label in zip(data, labels):
        if item > threshold:
            results.append((label, item))
    return results

# with type hints — crystal clear
def process_typed(
    data: list[float],
    threshold: float,
    labels: list[str],
) -> list[tuple[str, float]]:
    """Filter data points above threshold and pair with labels."""
    results: list[tuple[str, float]] = []
    for item, label in zip(data, labels):
        if item > threshold:
            results.append((label, item))
    return results

# usage
scores = [0.9, 0.3, 0.7, 0.1, 0.85]
names = ['cat', 'dog', 'bird', 'car', 'plane']
high_conf = process_typed(scores, threshold=0.5, labels=names)
print(f"High confidence predictions: {high_conf}")

High confidence predictions: [('cat', 0.9), ('bird', 0.7), ('plane', 0.85)]


In [ ]:
# advanced type hints used in AI codebases
from typing import Literal, TypeAlias

# type aliases (clean up complex signatures)
Embedding: TypeAlias = list[float]
BatchEmbeddings: TypeAlias = list[Embedding]
ModelName: TypeAlias = Literal["llama3", "mistral", "phi4"]

def embed_texts(
    texts: list[str],
    model: ModelName = "llama3",
    normalize: bool = True,
) -> BatchEmbeddings:
    """Generate embeddings for a batch of texts."""
    # Simulated embeddings
    return [[0.1 * len(t)] * 5 for t in texts]

result = embed_texts(["hello world", "AI is great"], model="mistral")
print(f"Embeddings shape: {len(result)} texts × {len(result[0])} dims")

Embeddings shape: 2 texts × 5 dims


### 5.2 Dataclasses — Structured Configuration

Dataclasses replace dictionaries for configuration — they're typed, validated, and IDE-friendly.

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for a language model."""
    model_name: str
    temperature: float = 0.7
    max_tokens: int = 512
    top_p: float = 0.9
    seed: int = 42
    stop_sequences: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        if not 0 <= self.temperature <= 2:
            raise ValueError(f"Temperature must be 0-2, got {self.temperature}")
        if self.max_tokens < 1:
            raise ValueError(f"max_tokens must be positive, got {self.max_tokens}")

@dataclass
class RAGConfig:
    """Configuration for a RAG pipeline."""
    llm: ModelConfig
    chunk_size: int = 512
    chunk_overlap: int = 50
    top_k: int = 5
    embedding_model: str = "BAAI/bge-small-en-v1.5"

# clean, typed configuration
config = RAGConfig(
    llm=ModelConfig(model_name="llama3.2:3b", temperature=0.3),
    chunk_size=1024,
    top_k=3,
)

print(f"LLM: {config.llm.model_name}")
print(f"Temperature: {config.llm.temperature}")
print(f"Chunk size: {config.chunk_size}")
print(f"Top-k retrieval: {config.top_k}")
print(f"\nFull config: {config}")

LLM: llama3.2:3b
Temperature: 0.3
Chunk size: 1024
Top-k retrieval: 3

Full config: RAGConfig(llm=ModelConfig(model_name='llama3.2:3b', temperature=0.3, max_tokens=512, top_p=0.9, seed=42, stop_sequences=[]), chunk_size=1024, chunk_overlap=50, top_k=3, embedding_model='BAAI/bge-small-en-v1.5')


### 5.3 Decorators — Reusable Cross-Cutting Concerns

Decorators wrap functions to add behavior (logging, timing, retries, caching). Every production AI system uses them.

In [ ]:
# timer decorator — measure function execution time
def timer(func: Callable) -> Callable:
    """Decorator that measures and prints execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️  {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

# retry decorator — essential for API calls
def retry(max_retries: int = 3, delay: float = 1.0):
    """Decorator that retries a function on failure."""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt < max_retries - 1:
                        print(f"  ⚠️ Attempt {attempt+1} failed: {e}. Retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        print(f"  ❌ All {max_retries} attempts failed.")
                        raise
        return wrapper
    return decorator

# cache decorator — avoid redundant LLM calls
def cache(func: Callable) -> Callable:
    """Simple in-memory cache for function results."""
    _cache: dict[str, Any] = {}
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        key = str((args, sorted(kwargs.items())))
        if key not in _cache:
            _cache[key] = func(*args, **kwargs)
            print(f"  📝 Cache MISS for {func.__name__}")
        else:
            print(f"  ✅ Cache HIT for {func.__name__}")
        return _cache[key]
    return wrapper

# using decorators
@timer
@cache
def expensive_embedding(text: str) -> list[float]:
    """Simulate an expensive embedding computation."""
    time.sleep(0.1)  # Simulate latency
    return [hash(text) % 100 / 100.0] * 5

# first call — cache miss
result1 = expensive_embedding("Hello world")
# second call — cache hit (instant)
result2 = expensive_embedding("Hello world")
print(f"Results match: {result1 == result2}")

  📝 Cache MISS for expensive_embedding
⏱️  expensive_embedding took 0.1003s
  ✅ Cache HIT for expensive_embedding
⏱️  expensive_embedding took 0.0000s
Results match: True


### 5.4 Generators — Memory-Efficient Data Pipelines

Generators process data lazily — one item at a time — instead of loading everything into memory. Critical for large datasets.

In [ ]:
# simulate a document processing pipeline for RAG
def load_documents(file_paths: list[str]) -> Generator[str, None, None]:
    """Lazily load documents one at a time."""
    for path in file_paths:
        # simulate reading a file
        yield f"Content of {path}: Lorem ipsum dolor sit amet, consectetur adipiscing elit. " * 10

def chunk_documents(
    documents: Iterator[str],
    chunk_size: int = 100,
    overlap: int = 20,
) -> Generator[dict[str, Any], None, None]:
    """Split documents into overlapping chunks."""
    for doc_id, doc in enumerate(documents):
        words = doc.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk_words = words[i:i + chunk_size]
            if chunk_words:
                yield {
                    "doc_id": doc_id,
                    "chunk_index": i // (chunk_size - overlap),
                    "text": " ".join(chunk_words),
                    "n_words": len(chunk_words),
                }

def create_embeddings(
    chunks: Iterator[dict[str, Any]],
) -> Generator[dict[str, Any], None, None]:
    """Add embeddings to each chunk."""
    for chunk in chunks:
        chunk["embedding"] = [hash(chunk["text"]) % 100 / 100.0] * 4
        yield chunk

# pipeline — everything is lazy, nothing loaded into memory at once
files = [f"doc_{i}.pdf" for i in range(5)]
pipeline = create_embeddings(chunk_documents(load_documents(files), chunk_size=30))

# process items one by one
for i, item in enumerate(pipeline):
    if i < 3:  # Show first 3
        print(f"Chunk {i}: doc={item['doc_id']}, words={item['n_words']}, "
              f"embedding={item['embedding'][:2]}...")
    elif i == 3:
        print("...")

print(f"\nTotal chunks processed: {i + 1}")
print("→ All processed lazily — memory usage stays constant regardless of corpus size!")

Chunk 0: doc=0, words=30, embedding=[0.27, 0.27]...
Chunk 1: doc=0, words=30, embedding=[0.42, 0.42]...
Chunk 2: doc=0, words=30, embedding=[0.38, 0.38]...
...

Total chunks processed: 55
→ All processed lazily — memory usage stays constant regardless of corpus size!


### 5.5 Async/Await — Concurrent API Calls

In agentic AI workflows, you often need to call multiple APIs concurrently (LLM + search + database). `asyncio` is essential.

In [44]:
# Simulated async AI operations
async def call_llm(prompt: str, delay: float = 0.5) -> str:
    """Simulate an LLM API call."""
    await asyncio.sleep(delay)
    return f"LLM response to: '{prompt[:30]}...'"

async def search_documents(query: str, delay: float = 0.3) -> list[str]:
    """Simulate a vector database search."""
    await asyncio.sleep(delay)
    return [f"Doc result for '{query}'"]

async def get_user_context(user_id: str, delay: float = 0.2) -> dict:
    """Simulate fetching user context."""
    await asyncio.sleep(delay)
    return {"user_id": user_id, "preferences": ["technical", "concise"]}

# Sequential (slow) — 1.0s total
async def sequential_pipeline(prompt: str) -> dict:
    start = time.perf_counter()
    docs = await search_documents(prompt)
    context = await get_user_context("user_123")
    response = await call_llm(prompt)
    elapsed = time.perf_counter() - start
    return {"response": response, "time": elapsed}

# Concurrent (fast) — max(0.5, 0.3, 0.2) = 0.5s total
async def concurrent_pipeline(prompt: str) -> dict:
    start = time.perf_counter()
    docs, context, response = await asyncio.gather(
        search_documents(prompt),
        get_user_context("user_123"),
        call_llm(prompt),
    )
    elapsed = time.perf_counter() - start
    return {"response": response, "time": elapsed}

# Run both pipelines
prompt = "Explain the Transformer architecture in simple terms"

seq_result = await sequential_pipeline(prompt)
con_result = await concurrent_pipeline(prompt)

print(f"Sequential: {seq_result['time']:.3f}s")
print(f"Concurrent: {con_result['time']:.3f}s")
print(f"Speedup: {seq_result['time'] / con_result['time']:.1f}x")
print(f"\n→ asyncio.gather() runs all calls in parallel — essential for agent tool calls!")

Sequential: 1.002s
Concurrent: 0.501s
Speedup: 2.0x

→ asyncio.gather() runs all calls in parallel — essential for agent tool calls!


### 5.6 Comprehensions & Functional Patterns

Concise, readable transformations that replace verbose loops.

In [45]:
# List comprehension — filter and transform
scores = [0.92, 0.45, 0.78, 0.31, 0.87, 0.15, 0.96]
labels = ['cat', 'dog', 'bird', 'car', 'plane', 'boat', 'fish']

# High-confidence predictions
confident = [(label, f"{score:.0%}") for label, score in zip(labels, scores) if score > 0.7]
print(f"High confidence: {confident}")

# Dictionary comprehension — build lookup tables
score_map = {label: score for label, score in zip(labels, scores)}
print(f"Score map: {score_map}")

# Set comprehension — unique values
tokens = ["the", "cat", "sat", "on", "the", "mat", "the"]
unique_tokens = {token for token in tokens}
print(f"Unique tokens: {unique_tokens}")

# Generator expression — lazy, memory-efficient
total_chars = sum(len(token) for token in tokens)  # No intermediate list!
print(f"Total characters: {total_chars}")

# Nested comprehension — flatten batches
batches = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
flat = [item for batch in batches for item in batch]
print(f"Flattened: {flat}")

High confidence: [('cat', '92%'), ('bird', '78%'), ('plane', '87%'), ('fish', '96%')]
Score map: {'cat': 0.92, 'dog': 0.45, 'bird': 0.78, 'car': 0.31, 'plane': 0.87, 'boat': 0.15, 'fish': 0.96}
Unique tokens: {'mat', 'cat', 'the', 'on', 'sat'}
Total characters: 20
Flattened: [1, 2, 3, 4, 5, 6, 7, 8, 9]


### 5.7 Context Managers — Resource Management

Context managers ensure resources (files, connections, GPU memory) are properly cleaned up.

In [46]:
import contextlib

# Custom context manager for timing code blocks
@contextlib.contextmanager
def timer_context(label: str = "Block"):
    """Time a block of code."""
    start = time.perf_counter()
    print(f"⏱️  Starting: {label}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"⏱️  Finished: {label} ({elapsed:.4f}s)")

# Custom context manager for temporary config changes
@contextlib.contextmanager
def override_config(config: ModelConfig, **overrides):
    """Temporarily override config values."""
    original = {k: getattr(config, k) for k in overrides}
    for k, v in overrides.items():
        setattr(config, k, v)
    try:
        yield config
    finally:
        for k, v in original.items():
            setattr(config, k, v)

# Usage
cfg = ModelConfig(model_name="llama3", temperature=0.7)

with timer_context("Processing"):
    time.sleep(0.1)
    with override_config(cfg, temperature=0.0, max_tokens=100) as temp_cfg:
        print(f"  Inside override: temp={temp_cfg.temperature}, max_tokens={temp_cfg.max_tokens}")
    print(f"  After override:  temp={cfg.temperature}, max_tokens={cfg.max_tokens}")

⏱️  Starting: Processing
  Inside override: temp=0.0, max_tokens=100
  After override:  temp=0.7, max_tokens=512
⏱️  Finished: Processing (0.1003s)


### 5.8 Enums & Error Handling Patterns

In [47]:
class MessageRole(str, Enum):
    """Chat message roles (used in every LLM API)."""
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"

@dataclass
class ChatMessage:
    """A single chat message."""
    role: MessageRole
    content: str

    def to_dict(self) -> dict[str, str]:
        return {"role": self.role.value, "content": self.content}

# Build a conversation
conversation: list[ChatMessage] = [
    ChatMessage(MessageRole.SYSTEM, "You are a helpful AI assistant."),
    ChatMessage(MessageRole.USER, "What is a Transformer?"),
    ChatMessage(MessageRole.ASSISTANT, "A Transformer is a neural network architecture..."),
]

# Serialize for API call
messages = [msg.to_dict() for msg in conversation]
print(json.dumps(messages, indent=2))

[
  {
    "role": "system",
    "content": "You are a helpful AI assistant."
  },
  {
    "role": "user",
    "content": "What is a Transformer?"
  },
  {
    "role": "assistant",
    "content": "A Transformer is a neural network architecture..."
  }
]


## 7. Evaluation — Build a Mini Tool-Calling Agent

In [48]:
# Mini agent framework using all concepts learned
@dataclass
class ToolResult:
    tool_name: str
    result: str
    success: bool

class MiniAgent:
    """A tiny agent framework demonstrating Python mastery concepts."""

    def __init__(self, name: str) -> None:
        self.name = name
        self._tools: dict[str, Callable] = {}
        self.history: list[dict[str, str]] = []

    def tool(self, func: Callable) -> Callable:
        """Decorator to register a function as an agent tool."""
        self._tools[func.__name__] = func
        return func

    def call_tool(self, tool_name: str, **kwargs) -> ToolResult:
        """Call a registered tool by name."""
        if tool_name not in self._tools:
            return ToolResult(tool_name, f"Unknown tool: {tool_name}", False)
        try:
            result = self._tools[tool_name](**kwargs)
            return ToolResult(tool_name, str(result), True)
        except Exception as e:
            return ToolResult(tool_name, str(e), False)

    def available_tools(self) -> list[str]:
        return list(self._tools.keys())

# Create an agent and register tools
agent = MiniAgent("ResearchBot")

@agent.tool
def calculate(expression: str) -> float:
    """Evaluate a math expression."""
    return eval(expression)  # In production, use a safe parser!

@agent.tool
def word_count(text: str) -> int:
    """Count words in text."""
    return len(text.split())

@agent.tool
def summarize(text: str, max_words: int = 10) -> str:
    """Return first N words as a summary."""
    words = text.split()
    return " ".join(words[:max_words]) + ("..." if len(words) > max_words else "")

# Use the agent
print(f"Agent: {agent.name}")
print(f"Available tools: {agent.available_tools()}\n")

# Call tools
results = [
    agent.call_tool("calculate", expression="2**10 + 42"),
    agent.call_tool("word_count", text="The Transformer architecture revolutionized NLP"),
    agent.call_tool("summarize", text="The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al in 2017", max_words=5),
    agent.call_tool("unknown_tool"),  # Error case
]

for r in results:
    status = "✅" if r.success else "❌"
    print(f"  {status} {r.tool_name}: {r.result}")

Agent: ResearchBot
Available tools: ['calculate', 'word_count', 'summarize']

  ✅ calculate: 1066
  ✅ word_count: 5
  ✅ summarize: The Transformer architecture was introduced...
  ❌ unknown_tool: Unknown tool: unknown_tool


## 8. Exercises

### Exercise 1: Build a Retry Decorator with Exponential Backoff
Modify the `retry` decorator to use exponential backoff (1s, 2s, 4s, 8s...) instead of a fixed delay.

### Exercise 2: Implement a Batched Generator
Write a generator that takes an iterable and yields batches of a specified size.

In [49]:
# Exercise 2 — Batched generator
from typing import TypeVar

T = TypeVar('T')

def batch_items(items: Iterator[T], batch_size: int) -> Generator[list[T], None, None]:
    """Yield successive batches from an iterator."""
    batch: list[T] = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:  # Don't forget the last partial batch
        yield batch

# Test
data = range(17)
for i, b in enumerate(batch_items(iter(data), batch_size=5)):
    print(f"  Batch {i}: {b}")

  Batch 0: [0, 1, 2, 3, 4]
  Batch 1: [5, 6, 7, 8, 9]
  Batch 2: [10, 11, 12, 13, 14]
  Batch 3: [15, 16]


## 9. Challenge Problems

### Challenge 1: Build an Async Rate Limiter
Implement an async rate limiter decorator that limits function calls to N per second (essential for API rate limits).

### Challenge 2: Plugin System
Build a plugin system using decorators and a registry pattern where users can register custom tools by simply decorating their functions.

### Challenge 3: Typed Configuration Loader
Build a config system that loads from YAML/JSON files into dataclasses, with validation and environment variable overrides.

## 10. Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Fluent Python, 3rd Ed](https://www.oreilly.com/library/view/fluent-python-2nd/9781492056348/) | 📘 Book | The definitive advanced Python reference |
| [Real Python — Async IO](https://realpython.com/async-io-python/) | 📖 Tutorial | Comprehensive asyncio guide |
| [mypy Documentation](https://mypy.readthedocs.io/) | 📖 Docs | Static type checking for Python |
| [Python Decorators](https://realpython.com/primer-on-python-decorators/) | 📖 Tutorial | In-depth decorator patterns |

---

**Next:** [05 — NumPy Essentials →](05-numpy-essentials.ipynb)